In [33]:
import pandas as pd
import numpy as np

# pip install category-encoders
import category_encoders as ce

# it seems this data doesn't follow the UTF-8 -format
# let's force the encoding to follow Latin1
df = pd.read_csv("OnlineRetail.csv", encoding="ISO-8859-1")

# a very simple interaction feature by MULTIPLICATION
# you can use same logic can be used with additions, substraction, division etc.
# just be careful you don't divide zero
df['Sales'] = df['Quantity'] * df['UnitPrice']
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France,10.20
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France,12.60
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France,16.60
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France,16.60


In [34]:
df.describe()

,Quantity,UnitPrice,CustomerID,Sales
count,541909.000000,541909.000000,406829.000000,541909.000000
mean,9.552250,4.611114,15287.690570,17.987795
std,218.081158,96.759853,1713.600303,378.810824
min,-80995.000000,-11062.060000,12346.000000,-168469.600000
25%,1.000000,1.250000,13953.000000,3.400000
50%,3.000000,2.080000,15152.000000,9.750000
75%,10.000000,4.130000,16791.000000,17.400000
max,80995.000000,38970.000000,18287.000000,168469.600000


In [35]:
len(df['Description'].unique())

4224

**Example 1: Target encoding (replace each unique category with that category's average)**

In [36]:
# create a target encoder by using category encoders
encoder = ce.TargetEncoder(cols=["Description"])

# apply the encoder, Description is going the be encoded, Sales is our target variable
df_encoded_target = encoder.fit_transform(df['Description'], df['Sales'])

# insert the encoded description averages back to the original data
df['Description_encoded'] = df_encoded_target

# the usual process is this => you save and use the original product name
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the sales => drop the original product name
# df.drop("Description", axis=1)
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales,Description_encoded
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30,42.071959
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,24.136433
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00,25.579590
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,34.050825
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,48.749510


In [37]:
# based on target encoding, average price for WHITE METAL LANTERN is 24.136433
df_test = df[df['Description'] == "WHITE METAL LANTERN"]
df_test['Sales'].mean()

# Sales is our target variable
# for each category in description => 
# we replace the text category with its average of the sales value

np.float64(24.13643292682927)

**If data overfits easily with ML model and the target encoded data (average), you can use LeaveOneOutEncoder instead to reduce it (adds a bit of variance)**

In [38]:
# create a target encoder by using category encoders
encoder = ce.LeaveOneOutEncoder(cols=["Description"])

# apply the encoder, Description is going the be encoded, Sales is our target variable
df_encoded_target = encoder.fit_transform(df['Description'], df['Sales'])

# insert the encoded description averages back to the original data
df['Description_LOO_encoded'] = df_encoded_target

# the usual process is this => you save and use the original product name
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the sales => drop the original product name
# df.drop("Description", axis=1)
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales,Description_encoded,Description_LOO_encoded
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30,42.071959,42.083264
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,24.136433,24.148043
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00,25.579590,25.591849
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,34.050825,34.079873
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,48.749510,48.812924


In [39]:
# Leave One Out -encoder is almost the same as 
# Target encoding BUT for each row in the dataset
# and the average target calculation, THAT ONE PARTICULAR
# row is excluded from the calculation
category_row_sales = [
    2, 
    5,
    #7,
    2,
    5,
    1
]

average = sum(category_row_sales) / len(category_row_sales)
average

3.0

**Example 2 - Frequency (count) encoder => replace each unique category with the frequency/count of that category (probably measures popularity of a category quite well, if that's important for your ML model**

In [40]:
# create a target encoder by using category encoders
encoder = ce.CountEncoder(cols=["Description"])

# apply the encoder, Description is going the be encoded
df_encoded_target = encoder.fit_transform(df['Description'])

# insert the encoded description averages back to the original data
df['Description_frequency_encoded'] = df_encoded_target

# the usual process is this => you save and use the original product name
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the sales => drop the original product name
# df.drop("Description", axis=1)
df[['Description', 'Sales', 'Description_frequency_encoded']].head(5)

# remember: if you drop the original description and remove all duplicates
# you will most likely have less rows because of that

# the frequency/coutn approach is probably most effective 
# when the popularity of some certain item has a meaning in ML model
# for sales data, popular items probably have more optimisitic sales estimations
# (etc. depends on context/situation however)

,Description,Sales,Description_frequency_encoded
0,WHITE HANGING HEART T-LIGHT HOLDER,15.30,2369
1,WHITE METAL LANTERN,20.34,328
2,CREAM CUPID HEARTS COAT HANGER,22.00,293
3,KNITTED UNION FLAG HOT WATER BOTTLE,20.34,473
4,RED WOOLLY HOTTIE WHITE HEART.,20.34,449


**Example 3 - Hash encoding / feature hashing (or hashing trick)**

In [41]:
# create a target encoder by using category encoders
encoder = ce.HashingEncoder(cols=["Description"], n_components=6)

# apply the encoder, Description is going the be encoded
df_encoded_hash = encoder.fit_transform(df)

# we have now simplified 4000+ categories into 6
# consult Google/generative AI for ideas on how to evaluate
# the quality of hashing and the amount of buckets
df_encoded_hash


,col_0,col_1,col_2,col_3,col_4,col_5,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales,Description_encoded,Description_LOO_encoded,Description_frequency_encoded
0,1,0,0,0,0,0,536365,85123A,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30,42.071959,42.083264,2369
1,0,0,1,0,0,0,536365,71053,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,24.136433,24.148043,328
2,0,0,0,1,0,0,536365,84406B,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00,25.579590,25.591849,293
3,0,0,0,1,0,0,536365,84029G,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,34.050825,34.079873,473
4,0,1,0,0,0,0,536365,84029E,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34,48.749510,48.812924,449
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541904,0,1,0,0,0,0,581587,22613,12,12/9/2011 12:50,0.85,12680.0,France,10.20,8.203473,8.189864,148
541905,0,0,0,0,1,0,581587,22899,6,12/9/2011 12:50,2.10,12680.0,France,12.60,13.493844,13.496646,320
541906,0,0,1,0,0,0,581587,23254,4,12/9/2011 12:50,4.15,12680.0,France,16.60,21.823322,21.840392,307
541907,1,0,0,0,0,0,581587,23255,4,12/9/2011 12:50,4.15,12680.0,France,16.60,25.086538,25.139255,162
